# Hello World from a GPU, inside a Phlex job

This notebook runs CUDA code from a Phlex algorithm. The CUDA part is the smallest GPU program
there is: launch **one GPU thread**, and that thread prints

```
Hello World from GPU thread 0!
```

When you finish, you will be able to:
- explain what a CUDA kernel, a thread and a launch configuration are
- put CUDA code in a library that Phlex code can call, without nvcc ever seeing a Phlex header
- write the `CMakeLists.txt` for a project that mixes C++ and CUDA
- run a Phlex job whose transform runs on the GPU, and check its result with an observer

## How this fits with the other workshop tutorials

- **Do `phlex-first-steps.ipynb` first**, and ideally `phlex-examples-tutorial.ipynb` up to Section 6.
  This notebook uses the ideas from there: providers, transforms, observers, `product_selector`,
  `PHLEX_PLUGIN_PATH`, and the **SHARED** versus **MODULE** library split.
- The key idea is the same one Kyle gave: **Phlex provides the higher-order functions; you provide
  the operators.** Here, the operator is an ordinary C++ function that happens to start work on a GPU.
- The companion notebook, `gpu-benchmark.ipynb`, uses the same structure to time real GPU work.

## The plan

```
 job ──► n_threads_source ──► n_threads ──► transform "gpu_hello" ──► greetings
                                  │              │ calls examples::gpu::say_hello()
                                  │              ▼ in libmy_gpu_hello.so (compiled by nvcc)
                                  │            GPU: "Hello World from GPU thread 0!"
                                  └──────────► observer "verify_greetings" ◄──┘
```

- **Provider:** `n_threads_source` provides how many GPU threads to launch (1) as a data product.
- **Transform:** `gpu_hello` calls the CUDA library, which launches the kernel and returns how many
  threads said hello.
- **Observer:** `verify_greetings` checks that every launched thread said hello.

# Section 0: Software setup

You need **phlex**, **CMake 3.31 or later**, a C++ compiler, the **CUDA toolkit** (`nvcc`) and a
**GPU** that the kernel can see.

**Check the kernel name at the top right: it should say "PHA India (Apptainer)".** If it says
**Python 3**, switch with **Kernel → Change Kernel…**. The Python 3 kernel runs outside the
container, where there is no phlex or cmake. The PHA India kernel runs inside the image, so every
`!` and `%%bash` cell below can see phlex, and it starts the container with `--nv`, so it can see the GPU.

Run the next cell to check that the tools and the GPU are visible. Run it again if you restart the kernel.

In [ ]:
!which phlex cmake c++ nvcc nvidia-smi
!cmake --version | head -1
!nvcc --version | tail -2
!phlex --version
!nvidia-smi -L || echo "nvidia-smi found no GPU"

What you should see:

- a path for every tool, and a `release` line from `nvcc`
- at least one line like `GPU 0: NVIDIA A100 ...` from `nvidia-smi -L`

If something is missing, you can still read the code below, but not run it on a GPU. There are two
separate things to check:

- **`nvidia-smi` missing or no GPU listed: the kernel was started without `--nv`.** The GPU driver and
  `nvidia-smi` come from the host, and apptainer only passes them into the container when it is
  started with `--nv`. You cannot add that flag from inside a notebook, because the kernel *is* the
  container: it has to be in the kernel's start command. Use the **PHA India (Apptainer) (GPU)** kernel if
  the kernel menu has one. If not, make your own copy of the kernel with `--nv`, once, in a terminal
  outside the container, from the folder this notebook is in:

  💻 **Terminal, outside the container** (prompt `jupyter-you@dune-tech`)
  ```bash
  python3 make_gpu_kernel.py
  ```

  Then reload this page, choose **PHA India (Apptainer) (GPU)** (top right) and run the cell above again.
  To check by hand first: `apptainer shell --nv --cleanenv <image>.sif`, then `nvidia-smi -L`.
- **`nvcc` missing: the image has no CUDA toolkit.** `--nv` does not help here: it brings the driver,
  not the compiler. The image needs CUDA added (see **Troubleshooting** at the end).

## Paths

You work in `~/phlex-work`, the same folder as the other Phlex tutorials, in a new
`phlex-gpu-examples` directory. The build directory sits next to the source directory, as in the
phlex-examples README:

```
~/phlex-work/phlex-gpu-examples/
├── hello/          <- source: the CUDA library, the Phlex plugins, CMakeLists.txt, the configuration
└── build-hello/    <- CMake build directory
```

Each `!` cell runs in a new shell, so `cd` or `export` in one cell does not carry over to the next.
The next cell sets the paths as environment variables of the kernel, which every later cell inherits.
It also chooses which GPU architecture to compile for. Run it again if you restart the kernel.

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

WORK = Path.home() / "phlex-work" / "phlex-gpu-examples"
GPU_SRC = WORK / "hello"
GPU_BUILD = WORK / "build-hello"
GPU_SRC.mkdir(parents=True, exist_ok=True)
GPU_BUILD.mkdir(parents=True, exist_ok=True)

os.environ["GPU_SRC"] = str(GPU_SRC)
os.environ["GPU_BUILD"] = str(GPU_BUILD)

# phlex finds plugins (MODULE libraries) through PHLEX_PLUGIN_PATH. Put the build directory first
# and keep what the kernel already has (that is where phlex's own plugins, like generate_layers, are).
entries = os.environ.get("PHLEX_PLUGIN_PATH", "").split(":")
if str(GPU_BUILD) not in entries:
    os.environ["PHLEX_PLUGIN_PATH"] = ":".join([str(GPU_BUILD)] + [e for e in entries if e])

# Compile the plugins with the g++ on PATH: the compiler Phlex was built with, and the one nvcc uses
# as its host compiler. Without this, CMake follows a CXX variable from the image, which can name
# a different compiler (such as clang++).
gxx = shutil.which("g++")
if gxx:
    os.environ["CXX"] = gxx

# Find nvcc: on PATH already, or in a CUDA toolkit mounted into the container at /usr/local/cuda*
if not shutil.which("nvcc"):
    for candidate in sorted(Path("/usr/local").glob("cuda*/bin/nvcc"), reverse=True):
        os.environ["PATH"] = f"{candidate.parent}:{os.environ['PATH']}"
        break
nvcc = shutil.which("nvcc")

# CUDA 12 refuses GCC 15 as nvcc's host compiler. If this CUDA is older than 13 and the image's
# compiler is GCC 15 or newer, let nvcc use the operating system's g++ for the .cu files instead.
# The Phlex plugins are still compiled by the image's GCC 15.
os.environ.setdefault("CUDA_CMAKE_ARGS", "")
if nvcc and not os.environ["CUDA_CMAKE_ARGS"]:
    release = subprocess.run([nvcc, "--version"], capture_output=True, text=True).stdout
    cuda_major = int(release.split("release ")[1].split(".")[0]) if "release " in release else 0
    gcc_major = int(subprocess.run(["c++", "-dumpversion"], capture_output=True, text=True).stdout.split(".")[0] or 0)
    system_gxx = Path("/usr/bin/g++")
    if cuda_major and cuda_major < 13 and gcc_major >= 15 and system_gxx.exists():
        os.environ["CUDA_CMAKE_ARGS"] = f"-DCMAKE_CUDA_HOST_COMPILER={system_gxx}"

# Which GPU architecture should nvcc compile for?
#   - a GPU is visible: 'native', the exact GPU(s) in this machine
#   - no GPU visible:   a list of common ones, so that the code at least compiles
gpus = ""
if shutil.which("nvidia-smi"):
    gpus = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip()
os.environ["CUDA_ARCH"] = "native" if gpus.startswith("GPU") else "75;80;86;89;90"

# To pass other CMake arguments for CUDA, set CUDA_CMAKE_ARGS before running this cell, for example
#   os.environ["CUDA_CMAKE_ARGS"] = "-DCMAKE_CUDA_FLAGS=-allow-unsupported-compiler"

for var in ["GPU_SRC", "GPU_BUILD", "PHLEX_PLUGIN_PATH", "CXX", "CUDA_ARCH", "CUDA_CMAKE_ARGS"]:
    print(f"{var:18} {os.environ.get(var, '')}")
print(f"{'nvcc':18} {nvcc or '⚠️  not found: the image has no CUDA toolkit, so the GPU code cannot be built'}")
print()
print(gpus or "⚠️  No GPU visible: you can build, but running the GPU code will fail.")
if not shutil.which("phlex"):
    print("⚠️  phlex not found: this notebook is not on the PHA India (Apptainer) kernel. Switch to it (top right) and run this cell again.")

# Section 1: A GPU in five minutes

A CPU has a few fast cores. A GPU has thousands of simple ones, built to run the **same code on
many pieces of data at once**. You write that code as a **kernel**: a function marked `__global__`
that the CPU (the **host**) launches on the GPU (the **device**).

```cpp
__global__ void hello_kernel() { printf("Hello World from GPU thread %d!\n", threadIdx.x); }

hello_kernel<<<1, 1>>>();   // <<<number of blocks, threads per block>>>
cudaDeviceSynchronize();    // wait for the GPU to finish
```

| Term | Meaning |
|---|---|
| **kernel** | A `__global__` function. One launch runs it once **per thread**. |
| **thread** | One copy of the kernel. `threadIdx.x` is its number inside its block. |
| **block** | A group of up to 1024 threads that run on the same part of the GPU. `blockIdx.x` numbers the blocks. |
| **launch configuration** | `<<<blocks, threads_per_block>>>`. `<<<1, 1>>>` is one block of one thread: a single GPU thread. |
| **asynchronous** | The launch returns at once, before the kernel has run. `cudaDeviceSynchronize()` waits. |

Two details matter for this example:

- `printf` on the GPU writes into a buffer on the device. The text reaches your screen only when the
  host **synchronizes** with the GPU. Forget `cudaDeviceSynchronize()` and the program can end before
  anything is printed.
- A kernel cannot return a value. To get an answer back, the kernel writes to **device memory**
  and the host copies it back with `cudaMemcpy`. Here each thread adds 1 to a counter, so the host
  can check that the thread really ran.

> 🧭 **Analogy:** the CPU is a lab manager and the GPU is a room of a thousand technicians. A kernel
> is one instruction sheet. The launch configuration says how many technicians get a copy. The manager
> hands out the sheets and goes back to work (asynchronous); to read the results, the manager has to
> wait for the room to finish (synchronize).

> 🔬 **In an analysis:** DUNE's optical photon simulation is a natural GPU job. Millions of photons
> follow the same rules independently, so each GPU thread can track one photon. The detector
> simulation chain on the CPU only sees the photons that reach a photon detector.

# Section 2: The CUDA library

The CUDA code lives in a **framework-agnostic library**, like `my_add` in phlex-examples. Two files:

- `my_gpu_hello.hpp`: the interface. It is **plain C++**: no `__global__`, no CUDA types. Any
  C++ code can include it, including a Phlex plugin compiled as C++23.
- `my_gpu_hello.cu`: the implementation, compiled by **nvcc**. It never includes a Phlex header.

This split means the CUDA compiler never has to understand Phlex's C++23 headers, and the Phlex
plugin never has to know about CUDA.

In [ ]:
%%writefile ~/phlex-work/phlex-gpu-examples/hello/my_gpu_hello.hpp
#ifndef MY_GPU_HELLO_HPP
#define MY_GPU_HELLO_HPP

// Plain C++ interface to the CUDA code in my_gpu_hello.cu.
//
// Nothing in this header is CUDA-specific, so the Phlex plugin that includes it is compiled by the
// ordinary C++ compiler (C++23, Phlex headers) and never by nvcc. Only my_gpu_hello.cu is compiled
// by nvcc. This is the same split as my_add.hpp / my_add.cpp in phlex-examples: a
// framework-agnostic library, plus a thin plugin that registers it with Phlex.

namespace examples::gpu {

  // Number of CUDA devices this process can see (0 if there is no GPU or no driver).
  int device_count();

  // Launch one block of n_threads GPU threads. Each thread prints
  //   Hello World from GPU thread <threadIdx.x>!
  // and increments a counter in device memory. Returns the counter, copied back to the host,
  // so the caller can check that every thread really ran.
  // Throws std::runtime_error if any CUDA call fails.
  int say_hello(int n_threads);

}

#endif // MY_GPU_HELLO_HPP

In [ ]:
%%writefile ~/phlex-work/phlex-gpu-examples/hello/my_gpu_hello.cu
#include "my_gpu_hello.hpp"

#include <cuda_runtime.h>

#include <cstdio>
#include <stdexcept>
#include <string>

namespace {

  // The kernel: runs on the GPU, once per thread.
  __global__ void hello_kernel(int* greeted)
  {
    printf("Hello World from GPU thread %d!\n", threadIdx.x);
    atomicAdd(greeted, 1); // count the threads that ran
  }

  void check(cudaError_t const err, char const* what)
  {
    if (err != cudaSuccess) {
      throw std::runtime_error(std::string{what} + " failed: " + cudaGetErrorString(err));
    }
  }

}

namespace examples::gpu {

  int device_count()
  {
    int n = 0;
    if (cudaGetDeviceCount(&n) != cudaSuccess) {
      return 0;
    }
    return n;
  }

  int say_hello(int const n_threads)
  {
    if (n_threads < 1 || n_threads > 1024) {
      throw std::invalid_argument("say_hello: one block holds 1 to 1024 threads, got " +
                                  std::to_string(n_threads));
    }

    int* d_greeted = nullptr;
    check(cudaMalloc(&d_greeted, sizeof(int)), "cudaMalloc");
    check(cudaMemset(d_greeted, 0, sizeof(int)), "cudaMemset");

    // <<<blocks, threads per block>>>: one block of n_threads threads
    hello_kernel<<<1, n_threads>>>(d_greeted);
    check(cudaGetLastError(), "kernel launch"); // bad launch configuration, no device, ...

    // Kernel launches are asynchronous. Waiting here also flushes the device-side printf buffer.
    check(cudaDeviceSynchronize(), "kernel execution");
    std::fflush(stdout);

    int greeted = 0;
    check(cudaMemcpy(&greeted, d_greeted, sizeof(int), cudaMemcpyDeviceToHost), "cudaMemcpy");
    check(cudaFree(d_greeted), "cudaFree");
    return greeted;
  }

}

What `say_hello` does, step by step:

| Step | Code | Why |
|---|---|---|
| Check the input | `n_threads < 1 \|\| n_threads > 1024` | One block holds at most 1024 threads |
| Allocate a counter on the GPU | `cudaMalloc`, `cudaMemset` | The kernel can only write to device memory |
| Launch | `hello_kernel<<<1, n_threads>>>(d_greeted)` | One block of `n_threads` threads |
| Check the launch | `cudaGetLastError()` | Catches a bad launch configuration, or no usable GPU |
| Wait | `cudaDeviceSynchronize()` | The kernel has finished, its errors are reported, and its `printf` text is printed |
| Copy the answer back | `cudaMemcpy(..., cudaMemcpyDeviceToHost)` | Now the host knows how many threads ran |

Every CUDA call returns an error code, and `check()` turns a failure into a C++ exception. A Phlex job
then stops and prints the error, instead of carrying on with a wrong answer.

## Try the library without Phlex first

A tiny program calls the same function. If this works, the GPU, the driver and the toolkit are
fine, and any later problem is in the Phlex part. Testing the library on its own is the point of keeping
it framework-agnostic.

In [ ]:
%%writefile ~/phlex-work/phlex-gpu-examples/hello/hello_standalone.cpp
// Run the CUDA library on its own, without Phlex: the quickest way to check the GPU works.
#include "my_gpu_hello.hpp"

#include <cstdio>
#include <exception>

int main()
{
  int const n_devices = examples::gpu::device_count();
  std::printf("CUDA devices visible: %d\n", n_devices);
  if (n_devices == 0) {
    std::printf("No GPU visible to this process. Nothing to launch.\n");
    return 1;
  }

  try {
    int const greeted = examples::gpu::say_hello(1); // a single GPU thread
    std::printf("Threads that said hello: %d\n", greeted);
    return greeted == 1 ? 0 : 1;
  }
  catch (std::exception const& e) {
    std::printf("CUDA error: %s\n", e.what());
    return 1;
  }
}

# Section 3: The Phlex plugins

Two MODULE libraries, the same pattern as `numbers_source` and `add_and_verify` in phlex-examples.

**The provider** makes the launch size a data product. It reads `n_threads` from the configuration
and provides it for every data cell of the configured layer.

In [ ]:
%%writefile ~/phlex-work/phlex-gpu-examples/hello/n_threads_source.cpp
#include "phlex/configuration.hpp"
#include "phlex/model/data_cell_index.hpp"
#include "phlex/source.hpp"

using namespace phlex;

// Provides the launch size as a data product: one "n_threads" for every data cell of the
// configured layer. With layer 'job' there is exactly one data cell, so one kernel launch.
PHLEX_REGISTER_PROVIDERS(m, config)
{
  auto const layer = config.get<std::string>("layer");
  auto const n_threads = config.get<int>("n_threads");

  m.provide("provide_n_threads", [n_threads](data_cell_index const&) -> int { return n_threads; })
    .output_product("input", "n_threads", experimental::identifier{layer});
}

**The algorithms.** Compare this with `add_and_verify.cpp`: it is the same shape. The only GPU-specific
choice is `concurrency::serial` on the transform.

In [ ]:
%%writefile ~/phlex-work/phlex-gpu-examples/hello/gpu_hello_and_verify.cpp
#include "my_gpu_hello.hpp"

#include "phlex/module.hpp"

#include <cassert>

PHLEX_REGISTER_ALGORITHMS(m, config)
{
  using namespace phlex;

  auto const layer = config.get<std::string>("layer");

  // Transform: n_threads in, number of threads that greeted out. The operator is an ordinary
  // C++ function from the CUDA library. concurrency::serial because there is one GPU: one
  // launch at a time keeps the printed lines of different launches from mixing.
  m.transform("gpu_hello", examples::gpu::say_hello, concurrency::serial)
    .input_family(product_selector{.creator = "input", .layer = layer, .suffix = "n_threads"})
    .output_product_suffixes("greetings");

  // Observer: every thread that was launched must have said hello.
  m.observe(
     "verify_greetings",
     [](int const n_threads, int const greetings) { assert(greetings == n_threads); },
     concurrency::unlimited)
    .input_family(product_selector{.creator = "input", .layer = layer, .suffix = "n_threads"},
                  product_selector{.creator = "gpu_hello", .layer = layer, .suffix = "greetings"});
}

- `m.transform("gpu_hello", examples::gpu::say_hello, ...)` registers the CUDA library's function
  directly as the operator. Phlex reads its signature, `int(int)`, to know it takes one `int` and makes one.
- `concurrency::serial` lets only one call run at a time. There is one GPU, and two launches at once
  would mix their printed lines. The GPU is a **shared resource**, like a single output file.
- The observer takes **two** products from the same data cell: `n_threads` from the provider and
  `greetings` from the transform. Phlex calls it once both exist.

> 🧭 **Analogy:** `concurrency::serial` on the GPU transform is a booking sheet for the one big
> microscope in the lab. Everyone gets their turn; nobody shares the eyepiece.

> 🔬 **In an analysis:** with many data cells, `serial` means the GPU work of different events
> waits in line while the CPU work of other algorithms carries on in parallel. That is often what you
> want: the GPU stays busy, and the CPU threads do everything else.

**The configuration.** No extra layers: the job is the only data cell, so the kernel is launched
once, with one thread.

In [ ]:
%%writefile ~/phlex-work/phlex-gpu-examples/hello/gpu-hello-workflow.jsonnet
{
  driver: {
    // No extra layers: the job is the only data cell, so the kernel is launched once.
    cpp: 'generate_layers',
  },
  sources: {
    n_threads_source: {
      cpp: 'n_threads_source',
      layer: 'job',
      n_threads: 1,  // a single GPU thread
    },
  },
  modules: {
    hello: {
      cpp: 'gpu_hello_and_verify',
      layer: 'job',
    },
  },
}

# Section 4: CMake for C++ and CUDA

In [ ]:
%%writefile ~/phlex-work/phlex-gpu-examples/hello/CMakeLists.txt
cmake_minimum_required(VERSION 3.31)

# Compile for the GPU(s) on this machine unless told otherwise, for example
#   cmake -DCMAKE_CUDA_ARCHITECTURES="80;90" ...
# 'native' needs a GPU visible when you run cmake. Must be set before project().
if(NOT DEFINED CMAKE_CUDA_ARCHITECTURES)
  set(CMAKE_CUDA_ARCHITECTURES native)
endif()

project(phlex_gpu_hello VERSION 0.0.1 LANGUAGES CXX CUDA)

set(CMAKE_CXX_STANDARD 23) # the Phlex plugins
set(CMAKE_CUDA_STANDARD 17) # the .cu file; nvcc never sees a Phlex header

find_package(Boost REQUIRED json)
find_package(TBB REQUIRED)
find_package(spdlog REQUIRED)
find_package(phlex REQUIRED)
find_package(CUDAToolkit REQUIRED)

# The same strict warnings as phlex-examples, for C++ only: nvcc rejects -Wpedantic and the
# -Wno-* flags, so every option is wrapped in a COMPILE_LANGUAGE:CXX generator expression.
if(CMAKE_CXX_COMPILER_ID STREQUAL "GNU")
  if(CMAKE_CXX_COMPILER_VERSION VERSION_GREATER_EQUAL "14.1")
    if(CMAKE_CXX_COMPILER_VERSION VERSION_LESS "15")
      add_compile_options($<$<COMPILE_LANGUAGE:CXX>:-Wno-stringop-overflow>)
    endif()
    if(CMAKE_CXX_COMPILER_VERSION VERSION_LESS "16")
      add_compile_options($<$<COMPILE_LANGUAGE:CXX>:-Wno-array-bounds>)
    endif()
  endif()
endif()
add_compile_options("$<$<COMPILE_LANGUAGE:CXX>:-Wall;-Wpedantic;-Werror>")

# Framework-agnostic experiment library, written in CUDA (like my_add, but on a GPU).
# N.B. The library type should be SHARED.
add_library(my_gpu_hello SHARED my_gpu_hello.cu)
# Link the shared CUDA runtime, and record where it is (RUNPATH): CMake counts the CUDA library
# directory as one of nvcc's own, so it would leave it out, and neither the linker nor phlex
# could then find libcudart.so when a program or plugin uses this library.
set_target_properties(my_gpu_hello PROPERTIES CUDA_RUNTIME_LIBRARY Shared BUILD_RPATH "${CUDAToolkit_LIBRARY_DIR}")
target_link_libraries(my_gpu_hello PRIVATE CUDA::cudart)

# The same library without Phlex: a plain program
add_executable(hello_standalone hello_standalone.cpp)
target_link_libraries(hello_standalone PRIVATE my_gpu_hello)

# Phlex plugins (MODULE libraries, loaded by phlex at run time)
add_library(n_threads_source MODULE n_threads_source.cpp)
target_link_libraries(n_threads_source PRIVATE phlex::module)

add_library(gpu_hello_and_verify MODULE gpu_hello_and_verify.cpp)
target_link_libraries(gpu_hello_and_verify PRIVATE phlex::module my_gpu_hello)
# Keep assert() active in a Release build: the observer checks with assert
target_compile_options(gpu_hello_and_verify PRIVATE -UNDEBUG)

New things compared with the phlex-examples `CMakeLists.txt`:

| Line | What it does |
|---|---|
| `set(CMAKE_CUDA_ARCHITECTURES native)` before `project()` | Compile for the GPU in this machine. The setup cell passes `-DCMAKE_CUDA_ARCHITECTURES` explicitly, which overrides it |
| `project(... LANGUAGES CXX CUDA)` | CMake finds `nvcc` as well as the C++ compiler, and compiles `.cu` files with it |
| `set(CMAKE_CUDA_STANDARD 17)` | The `.cu` file is C++17. Only the plugins, compiled by g++, need C++23 |
| `find_package(CUDAToolkit REQUIRED)` | Provides the target `CUDA::cudart`, the CUDA runtime library |
| `$<$<COMPILE_LANGUAGE:CXX>:...>` | A **generator expression**: apply `-Wall -Wpedantic -Werror` to C++ files only. nvcc does not accept `-Wpedantic`, so phlex-examples' plain `add_compile_options` would break the CUDA build |
| `add_library(my_gpu_hello SHARED my_gpu_hello.cu)` | The CUDA library. CMake sees `.cu` and uses nvcc |
| `CUDA_RUNTIME_LIBRARY Shared` | Link the shared CUDA runtime, so every library in the process uses the same one |
| `target_link_libraries(gpu_hello_and_verify PRIVATE phlex::module my_gpu_hello)` | The plugin links the CUDA library, exactly like `add_and_verify` links `my_add` |

# Section 5: Build

In [ ]:
%%bash
command -v phlex >/dev/null || { echo "⚠️  phlex not found: this is not the PHA India (Apptainer) kernel. Pick it (top right), then run Section 0 again."; exit 1; }
cd "$GPU_BUILD"
command -v nvcc >/dev/null || { echo "⚠️  nvcc not found. Run the setup cell in Section 0 and read its nvcc line."; exit 1; }
rm -f CMakeCache.txt  # configure from scratch, so the compilers chosen in Section 0 are used
cmake "$GPU_SRC" -DCMAKE_BUILD_TYPE=Release -DCMAKE_CUDA_ARCHITECTURES="$CUDA_ARCH" $CUDA_CMAKE_ARGS

Look for `The CUDA compiler identification is NVIDIA ...` near the top and `Build files have been
written to: …` at the end. If configure fails, see **Troubleshooting** at the end. Now build. The hub is
shared, so please use no more than 4 jobs.

In [ ]:
%%bash
command -v phlex >/dev/null || { echo "⚠️  phlex not found: this is not the PHA India (Apptainer) kernel. Pick it (top right), then run Section 0 again."; exit 1; }
cd "$GPU_BUILD"
cmake --build . -j 4
echo "=== Libraries and programs"
ls *.so hello_standalone

`libmy_gpu_hello.so` was compiled by nvcc; the other two libraries are Phlex plugins compiled by
the C++ compiler.

# Section 6: Run

First the library on its own:

In [ ]:
%%bash
command -v phlex >/dev/null || { echo "⚠️  phlex not found: this is not the PHA India (Apptainer) kernel. Pick it (top right), then run Section 0 again."; exit 1; }
cd "$GPU_BUILD"
./hello_standalone

Expected output:

```
CUDA devices visible: 1
Hello World from GPU thread 0!
Threads that said hello: 1
```

Now the same function inside a Phlex job. As in the other tutorials, pass a stage name with `--stage`.

In [ ]:
%%bash
command -v phlex >/dev/null || { echo "⚠️  phlex not found: this is not the PHA India (Apptainer) kernel. Pick it (top right), then run Section 0 again."; exit 1; }
cd "$GPU_BUILD"
phlex -c "$GPU_SRC/gpu-hello-workflow.jsonnet" --stage gpu -j 4

You should see:

```
Using configuration file: .../gpu-hello-workflow.jsonnet
[...] [info] Number of worker threads: 4
Hello World from GPU thread 0!
[...] [info]
Seen layers:

  job
```

(Older Phlex releases print `Processed layers:` instead of `Seen layers:`.) The job ends with exit code
0, which means the observer's `assert(greetings == n_threads)` passed.


## Troubleshooting

| What you see | What it means | What to do |
|---|---|---|
| `nvidia-smi` missing, or `No devices were found` | The kernel's container was started without `--nv` | Choose the **(GPU)** kernel, or run `python3 make_gpu_kernel.py` in a terminal outside the container (Section 0), reload the page and choose the new kernel |
| `which nvcc` prints nothing, even with the GPU kernel, or CMake says `Failed to find nvcc` | The image has no CUDA toolkit; `--nv` only brings the driver | Ask your hub admin to add CUDA to the image, or to mount the host's CUDA toolkit into the kernel at `/usr/local/cuda*` (the setup cell finds it there). CUDA 13 or later works with the image's GCC 15 and needs a 580-series or newer driver; with CUDA 12 the setup cell makes nvcc use the system g++ |
| cmake stops because no GPU was detected for `CMAKE_CUDA_ARCHITECTURES` `native` | You ran cmake without a visible GPU | Rerun the setup cell (it then picks a list of architectures), or fix the GPU visibility first |
| `unsupported GNU version! gcc versions later than ... are not supported` | Your C++ compiler is newer than this CUDA release supports | Point nvcc at an older g++ with `-DCMAKE_CUDA_HOST_COMPILER=/path/to/g++-14`, or, at your own risk, `-DCMAKE_CUDA_FLAGS=-allow-unsupported-compiler`. Put it in `CUDA_CMAKE_ARGS` in the setup cell, delete the build directory and configure again |
| `CUDA driver version is insufficient for CUDA runtime version` | No driver is visible, or the driver is older than the toolkit | Check `nvidia-smi` in this notebook. If it works, the toolkit is newer than the driver: use a matching toolkit |
| `no kernel image is available for execution on the device` | The code was compiled for another GPU architecture | Configure again with a visible GPU (`native`), or add your GPU's architecture to `CMAKE_CUDA_ARCHITECTURES` |
| `Could not locate library with specification '...'` | phlex cannot find a plugin | Run the setup cell again: it puts the build directory on `PHLEX_PLUGIN_PATH` |
| `No stage name was specified` | Current phlex needs a stage | Keep `--stage gpu` on the `phlex` command line |

## Summary

| Piece | File | Compiled by |
|---|---|---|
| Kernel and host wrapper | `my_gpu_hello.cu` | nvcc |
| Plain C++ interface | `my_gpu_hello.hpp` | included by both |
| Provider | `n_threads_source.cpp` | C++ compiler, C++23 |
| Transform and observer | `gpu_hello_and_verify.cpp` | C++ compiler, C++23 |
| Job | `gpu-hello-workflow.jsonnet` | read by phlex |

The rule to take away: **put the CUDA in a SHARED library with a plain C++ interface, and register
its functions from a thin MODULE**, exactly as phlex-examples does for CPU code. Mark the GPU
algorithm `concurrency::serial` unless you have a reason not to.

Next: `gpu-benchmark.ipynb` uses the same layout to **time** GPU work, and shows when moving work to
a GPU pays off and when it does not.